# Jupyter Notebook for Cleaning Compiled Mouse Human GAPDH.xlsx Import

## Import Libraries

In [101]:
import pandas as pd
import numpy as np
from collections import Counter
import os
from pathlib import Path
import re

## Functions

In [111]:
## To differentiate columns when they have the same names and strip them of any extra spaces
def clean_column_names(columns):
    counts = Counter()
    cleaned_columns = []

    for col in columns:
        if isinstance(col, str):
            # Remove leading/trailing spaces and collapse internal spaces
            clean_col = re.sub(r'\s+', ' ', col.strip()).title()
        else:
            clean_col = col

        counts[clean_col] += 1

        if counts[clean_col] == 1:
            cleaned_columns.append(clean_col)
        else:
            cleaned_columns.append(f"{clean_col}_{counts[clean_col]}")

    return cleaned_columns

## Cleaning Column Inputs
### Consistent Capitalizing of  'Experiment #' and 'Type' values
### Sample Name values with consistent naming scheme and capitalizing
def clean_dataframe_values(df):

    df = df.copy()

    # 1. Capitalize alphabetic characters in Experiment #
    if 'Experiment #' in df.columns:
        df['Experiment #'] = (
            df['Experiment #']
            .astype('string')
            .str.upper()
        )

    # 2. Title case values in Type
    if 'Type' in df.columns:
        df['Type'] = (
            df['Type']
            .astype('string')
            .str.strip()
            .str.title()
        )

    # 3. Standardize Sample Name to "Sample __"
    if 'Sample Name' in df.columns:
        def standardize_sample_name(value):
            if pd.isna(value):
                return value

            value = str(value).strip()

            # Extract the number following "Sample"
            match = re.search(r'\bsample\s*(\d+)\b', value, re.IGNORECASE)

            if match:
                return f"Sample {match.group(1)}"

            return value.title()

        df['Sample Name'] = df['Sample Name'].apply(standardize_sample_name)

    return df

## Define data path and import files

In [103]:
current_dir = Path.cwd()
print(current_dir)
# Get the directory of the current script, then its parent
parent_dir = Path.cwd().parent

/Users/dkaur/Documents/BioHackathon/KIDS26-Team11/scripts


In [104]:
data_path = f'{parent_dir}/dataset'
filename = 'Compiled Mouse Human GAPDH.xlsx'

In [105]:
all_sheets = pd.read_excel(f'{data_path}/{filename}', sheet_name=None)

## Dataframe Cleaning (Universal)

### Strip of rows that are completely blank, assign title format to headers and strip them of extra characters, and differentiate columns with same name

In [106]:
all_sheets_cleanlines = {}
all_sheets_cleanlines = {
    sheet_name: (
        df.dropna(how='all')
          .reset_index(drop=True)
          .set_axis(clean_column_names(df.columns), axis=1)
    )
    for sheet_name, df in all_sheets.items()
}

### Standardize Experiment #, Type, and Sample Name columns (Described in Function Above)

In [115]:
all_sheets_cleanlines = {
    sheet_name: clean_dataframe_values(df)
    for sheet_name, df in all_sheets_cleanlines.items()
}

### Each Single Sheet Corresponds to a Dataframe in the Dictionary Now

In [107]:
all_sheets_cleanlines.keys() #Sheet Names

dict_keys(['EW', 'NBL', 'OST', 'RBL', 'WT', 'Media Experiments', 'OST - backup tab', 'RMS'])

## DataFrame Specific Cleaning

### 'Media Experiments' is divided into three different tables each for a different 'NM Exp #' from left to right. I annotate the rows with the 'NM Exp #' and then concatenate the rows together to form a single table without repeating headers 

In [140]:
print(all_sheets_cleanlines['Media Experiments'].keys())

# Section 1:
Med_Exp_df1 = all_sheets_cleanlines['Media Experiments'][['Experiment #', 'Sample Name', 'Type', 'Sample Name.1', 'Target Name','Mean Equivalent Cq', 'Human/Mouse Ratio (Ct)', 'Mouse Contamination','Unnamed: 8']]
Med_Exp_df1 = Med_Exp_df1.rename(columns={'Unnamed: 8': 'NM Exp #'})
Med_Exp_df1 = Med_Exp_df1.dropna(how='all').reset_index(drop=True)
Med_Exp_df1['NM Exp #'] = Med_Exp_df1['NM Exp #'][0]

#Section 2:
Med_Exp_df2 = all_sheets_cleanlines['Media Experiments'][['Experiment #.1', 'Sample Name.2', 'Type.1','Sample Name.3', 'Target Name.1', 'Mean Equivalent Cq.1','Human/Mouse Ratio (Ct).1', 'Mouse Contamination.1', 'Unnamed: 17']]
Med_Exp_df2 = Med_Exp_df2.rename(columns={'Experiment #.1': 'Experiment #', 'Sample Name.2': 'Sample Name', 'Type.1': 'Type','Sample Name.3': 'Sample Name.1', 'Target Name.1': 'Target Name', 'Mean Equivalent Cq.1': 'Mean Equivalent Cq','Human/Mouse Ratio (Ct).1': 'Human/Mouse Ratio (Ct)', 'Mouse Contamination.1': 'Mouse Contamination', 'Unnamed: 17': 'NM Exp #'})
Med_Exp_df2 = Med_Exp_df2.dropna(how='all').reset_index(drop=True)
Med_Exp_df2['NM Exp #'] = Med_Exp_df2['NM Exp #'][0]

#Section 3:
Med_Exp_df3 = all_sheets_cleanlines['Media Experiments'][['Experiment #.2', 'Sample Name.4', 'Type.2', 'Sample Name.5','Target Name.2', 'Mean Equivalent Cq.2', 'Human/Mouse Ratio (Ct).2','Mouse Contamination.2', 'Unnamed: 26']]
Med_Exp_df3 = Med_Exp_df3.rename(columns={'Experiment #.2': 'Experiment #', 'Sample Name.4': 'Sample Name', 'Type.2': 'Type','Sample Name.5': 'Sample Name.1', 'Target Name.2': 'Target Name', 'Mean Equivalent Cq.2': 'Mean Equivalent Cq','Human/Mouse Ratio (Ct).2': 'Human/Mouse Ratio (Ct)', 'Mouse Contamination.2': 'Mouse Contamination', 'Unnamed: 26': 'NM Exp #'})
Med_Exp_df3 = Med_Exp_df3.dropna(how='all').reset_index(drop=True)
Med_Exp_df3['NM Exp #'] = Med_Exp_df3['NM Exp #'][0]

pd.concat([Med_Exp_df1,Med_Exp_df2,Med_Exp_df3]).reset_index(drop=True)

Index(['Experiment #', 'Sample Name', 'Type', 'Sample Name.1', 'Target Name',
       'Mean Equivalent Cq', 'Human/Mouse Ratio (Ct)', 'Mouse Contamination',
       'Unnamed: 8', 'Experiment #.1', 'Sample Name.2', 'Type.1',
       'Sample Name.3', 'Target Name.1', 'Mean Equivalent Cq.1',
       'Human/Mouse Ratio (Ct).1', 'Mouse Contamination.1', 'Unnamed: 17',
       'Experiment #.2', 'Sample Name.4', 'Type.2', 'Sample Name.5',
       'Target Name.2', 'Mean Equivalent Cq.2', 'Human/Mouse Ratio (Ct).2',
       'Mouse Contamination.2', 'Unnamed: 26'],
      dtype='object')


,Experiment #,Sample Name,Type,Sample Name.1,Target Name,Mean Equivalent Cq,Human/Mouse Ratio (Ct),Mouse Contamination,NM Exp #
0,OST 01.21.001,Mast 152 S7 (Os3),Organoids,Sample 1,hGAPDH,38,2.337023,2.686583,NM_8.22.26
1,<NA>,NaN,<NA>,Sample 1,mGapdh,16.26,NaN,NaN,NM_8.22.26
2,OST 01.21.001,Mast 152 S7 (Os1),Organoids,Sample 2,hGAPDH,38.252,2.328038,2.669305,NM_8.22.26
3,<NA>,NaN,<NA>,Sample 2,mGapdh,16.431,NaN,NaN,NM_8.22.26
4,OST 01.21.001,Mast 152 S7 (Os5),Organoids,Sample 3,hGAPDH,37.601,2.269632,2.556985,NM_8.22.26
...,...,...,...,...,...,...,...,...,...
133,NaN,NaN,NaN,Sample 22,mGapdh,18.004,NaN,NaN,NM_8.22.29
134,RBL 01.22.012,SJRB 165 p1 (3DRDM),suspension,Sample 23,hGAPDH,31.873,1.216620,0.531961,NM_8.22.29
135,NaN,NaN,NaN,Sample 23,mGapdh,26.198,NaN,NaN,NM_8.22.29
136,RBL 01.22.012,SJRB 165 p1 (3DRDM),adherent,Sample 24,hGAPDH,33.981,1.879376,1.806493,NM_8.22.29


## Final Cleaned DataFrames: Clean_dataframes
### Need Further Input on Renaming the 4 Consistent Columns between Sheets: [Experiment #, Sample Name, Type, Sample]
### AND how to Concatenate Data Between Sheets

In [151]:
#Store all Dataframes from Sheets:
Clean_dataframes = all_sheets_cleanlines

# Assign Individually Cleaned Dataframe:
Clean_dataframes['Media Experiments'] = pd.concat([Med_Exp_df1,Med_Exp_df2,Med_Exp_df3]).reset_index(drop=True)

## Notes/Questions from Inspection of each df
## 'EW', 'NBL', 'OST', 'RBL', 'WT', 'OST - backup tab', and 'RMS':
### Q1) Should I combine rows for the same [Experiment #, Sample Name, Type, Sample] with columns for 'Mouse Target', 'Human Target', 'Mouse Mean Equivalent Cq', and 'Human Mean Equivalent Cq'? --> <span style="color:red"> NO. Sometimes they have separate 'Sample Name 2' values </span>
### Follow-up Q1) Should I populate [Experiment #	Sample Name	Type] for the mice rows with the same values as the human row right above them? --> <span style="color:blue"> HAVE NOT YET. </span>
### Q2) Change the second 'Sample name' and Exp# to 'Collected Sample Name (EWS)' and 'EWS Exp#' or something? --> <span style="color:blue"> For the time being they are distinguished in each df by adding an iterative number to the end. </span>

## 'OST':
### Q3) What are the values for the columns [Experiment #, Sample Type, Sample Name, Target Name, Mean Equivalent Cq, Human/Mouse Ratio (Ct), Mouse Contamination, Nm Exp#, Injection Method] below which only have values for the last 4 columns? <span style="color:red">These might be misaligned:</span>

In [99]:
OST_df = all_sheets_cleanlines['OST']
OST_df[OST_df['Sample Name'].isna()]

,Experiment #,Sample,Type,Sample Name,Target Name,Mean Equivalent Cq,Human/Mouse Ratio (Ct),Mouse Contamination,Nm Exp#,Injection Method,Sample Name 2,Exp#,Ppx Submission Date,Ppx Result
66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MAST 586 p2 (OS) - PASSED,OST 01.22.013,2022-09-21,PASSED
137,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MAST 597 T - PASSED,OST 01.22.029,2022-09-21,PASSED


## Saving Each Cleaned Sheet as a separate csv file

In [152]:
# Create an output directory
output_dir = "Clean_GAPDH_dataframes"
os.makedirs(f'{parent_dir}/dataset/{output_dir}', exist_ok=True)

# Save each DataFrame as a separate CSV
for sheet_name, df in Clean_dataframes.items():
    # Clean the sheet name for use as a filename
    filename = f"{sheet_name}.csv"

    filepath = os.path.join(f'{parent_dir}/dataset/{output_dir}', filename)

    df.to_csv(filepath, index=False)

    print(f"Saved: {filepath}")

Saved: /Users/dkaur/Documents/BioHackathon/KIDS26-Team11/dataset/Clean_GAPDH_dataframes/EW.csv
Saved: /Users/dkaur/Documents/BioHackathon/KIDS26-Team11/dataset/Clean_GAPDH_dataframes/NBL.csv
Saved: /Users/dkaur/Documents/BioHackathon/KIDS26-Team11/dataset/Clean_GAPDH_dataframes/OST.csv
Saved: /Users/dkaur/Documents/BioHackathon/KIDS26-Team11/dataset/Clean_GAPDH_dataframes/RBL.csv
Saved: /Users/dkaur/Documents/BioHackathon/KIDS26-Team11/dataset/Clean_GAPDH_dataframes/WT.csv
Saved: /Users/dkaur/Documents/BioHackathon/KIDS26-Team11/dataset/Clean_GAPDH_dataframes/Media Experiments.csv
Saved: /Users/dkaur/Documents/BioHackathon/KIDS26-Team11/dataset/Clean_GAPDH_dataframes/OST - backup tab.csv
Saved: /Users/dkaur/Documents/BioHackathon/KIDS26-Team11/dataset/Clean_GAPDH_dataframes/RMS.csv
